# ESG News Intelligence: Reproducible Walkthrough

## tl;dr

This notebook runs the complete offline pipeline on synthetic company news: input validation, near-duplicate removal, ESG classification, sentiment scoring, and pillar summaries. The score is a media signal, not a standardized ESG rating.

## Context & Methods

The workflow implements the ESG and news-analysis idea in the ISA project report with an auditable, laptop-friendly baseline.

### Key Assumptions

- Titles and descriptions contain enough context for a first-pass classification.
- One article may belong to multiple ESG pillars.
- Similarity at or above 0.90 indicates a near-duplicate.
- News sentiment measures coverage tone, not underlying corporate performance.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from esg_news_agent.pipeline import analyze_articles
from esg_news_agent.sources import load_csv

## Data

The included CSV contains fictional articles so this walkthrough is deterministic and does not require credentials.

In [ ]:
sample_path = ROOT / "examples" / "sample_news.csv"
articles = load_csv(sample_path)
result = analyze_articles(articles, duplicate_threshold=0.90)
result["counts"]

### Validate Inputs and Pipeline Invariants

In [ ]:
assert result["counts"]["collected"] == 10
assert result["counts"]["duplicates_removed"] == 1
assert result["counts"]["esg_relevant"] == 8
assert all(0 <= item["media_signal_score"] <= 100 for item in result["pillars"].values())
"All checks passed"

## Results

Compare coverage volume, tone, and media-signal scores across the three ESG pillars.

In [ ]:
pillar_rows = [
    {"pillar": pillar, **metrics}
    for pillar, metrics in result["pillars"].items()
]
pillar_rows

### Inspect Article-Level Evidence

In [ ]:
[
    {
        "title": row["title"],
        "pillars": row["pillars"],
        "sentiment": row["sentiment_label"],
        "score": row["sentiment_score"],
    }
    for row in result["articles"]
]

## Takeaways

- The pipeline keeps the analysis traceable by exposing matched terms for every article.
- Duplicate removal prevents repeated syndication from dominating the summary.
- Pillar scores should be read together with article counts and source evidence.
- A validated transformer model can replace the baseline classifier without changing the collection, deduplication, or reporting interfaces.